### This notebook format data for training deepSCENIC from a SCENIC+ object

In [1]:
import pickle
import scanpy as sc
import numpy as np
import pandas as pd
import os

Read the SCENIC+ object

In [2]:
import dill
scplus_folder = "/lustre1/project/stg_00002/lcb/gpartel/projects/deepSCENIC/data/MM_lines"
scenicpls_obj = dill.load(
        open(os.path.join(scplus_folder, 'scplus_obj.pkl'), 'rb'))

/staging/leuven/stg_00002/lcb/gpartel/software/anaconda3/envs/DeepSEM/lib/python3.8/site-packages/requests/__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.9) or chardet (5.1.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [3]:
# PATH WHERE TO SAVE THE DATA
DATA_PATH = '/lustre1/project/stg_00002/lcb/gpartel/projects/deepSCENIC/data/MM_lines/'

# Save cell metadata
scenicpls_obj.metadata_cell.to_csv(DATA_PATH+'cells_metadata.csv')

Extract RNA and ATAC data, preprocessing and filtering

In [4]:
ad_rna = sc.AnnData(scenicpls_obj.to_df('EXP'))
ad_rna

/tmp/ipykernel_1014941/4235952067.py:1: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  ad_rna = sc.AnnData(scenicpls_obj.to_df('EXP'))


AnnData object with n_obs × n_vars = 936 × 17085

In [5]:
TFs = pd.read_csv("/staging/leuven/stg_00002/lcb/gpartel/software/DeepSEM_multimodal/resources/allTFs_hg38.txt", header=None)[0].to_list()

In [6]:
sc.pp.filter_genes(ad_rna, min_cells=10)
ad_rna

AnnData object with n_obs × n_vars = 936 × 15684
    var: 'n_cells'

In [7]:
sc.pp.normalize_total(ad_rna)
sc.pp.log1p(ad_rna)
ad_rna.layers['log_norm'] = ad_rna.X

In [8]:
# Subset TFs present in RNA data
TFs = ad_rna.var_names[ad_rna.var_names.isin(TFs)]
len(TFs)

1386

In [9]:
# Filter regions based on minimum number of fragments
# min_nr_acc = 20
# scenicpls_obj.subset(regions = scenicpls_obj.metadata_regions[scenicpls_obj.metadata_regions.cisTopic_nr_acc>min_nr_acc].index.to_list())

In [10]:
ad_atac = sc.AnnData(scenicpls_obj.to_df('ACC').T)
ad_atac

/tmp/ipykernel_1014941/3854231756.py:1: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  ad_atac = sc.AnnData(scenicpls_obj.to_df('ACC').T)


AnnData object with n_obs × n_vars = 936 × 288643

In [11]:
scenicpls_obj.region_names.values

array(['chr6:146944773-146945273', 'chr15:48634866-48635366',
       'chr19:16814090-16814590', ..., 'chrY:8126098-8126598',
       'chrY:9973540-9974040', 'chrY:9997664-9998164'], dtype=object)

Get regions per gene (up to 150 kb up and down stream of the gene)

In [12]:
from pycisTopic.utils import region_names_to_coordinates, coord_to_region_names
from scenicplus.enhancer_to_gene import get_search_space
import pyranges as pr

max_dist = 1000000
search_space = get_search_space(scenicpls_obj, pr_annot = pr.PyRanges(region_names_to_coordinates(scenicpls_obj.region_names.values)),
                                species = 'hsapiens',
                                assembly = 'hg38',
                                upstream = [max_dist, max_dist],
                                downstream = [max_dist, max_dist],
                                inplace=False,
                               )

2024-01-28 18:33:38,514 R2G          INFO     Downloading gene annotation from biomart dataset: hsapiens_gene_ensembl
2024-01-28 18:33:56,112 R2G          INFO     Downloading chromosome sizes from: http://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes
2024-01-28 18:33:57,675 R2G          INFO     Extending promoter annotation to 10 bp upstream and 10 downstream
2024-01-28 18:34:00,197 R2G          INFO     Extending search space to:
            						1000000 bp downstream of the end of the gene.
            						1000000 bp upstream of the start of the gene.
2024-01-28 18:34:14,647 R2G          INFO     Intersecting with regions.


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


2024-01-28 18:34:17,240 R2G          INFO     Calculating distances from region to gene
2024-01-28 18:40:54,780 R2G          INFO     Imploding multiple entries per region and gene
2024-01-28 18:59:20,227 R2G          INFO     Done!


Building region to gene distance matrix

In [13]:
search_space['Abs_distance'] = np.array([abs(x[0]) + 1 for x in search_space['Distance'].to_list()]).astype(int)

In [14]:
search_space_subset = search_space.loc[[gene in ad_rna.var_names for gene in search_space['Gene']]]

In [15]:
search_space_subset.shape

(3371674, 4)

In [16]:
search_space_subset_test = search_space_subset[(search_space_subset.Name.str.startswith('chr18'))]
regions_test = search_space_subset_test.loc[:, 'Name'].unique().tolist()
genes_test = search_space_subset_test.loc[:, 'Gene'].unique().tolist()

In [17]:
search_space_subset_train = search_space_subset[~((search_space_subset.Name.str.startswith('chr18')))]
regions_train = search_space_subset_train.loc[:, 'Name'].unique().tolist()
genes_train = search_space_subset_train.loc[:, 'Gene'].unique().tolist()

In [18]:
search_space_subset_test = search_space_subset_test[search_space_subset_test.Name.isin(regions_test)]
search_space_subset_train = search_space_subset_train[search_space_subset_train.Name.isin(regions_train)]

In [19]:
import math

def gaussian_function(x):
    mean=0
    std_dev = 100000
    exponent = -(1/2) * ((x - mean) / std_dev)**2
    return (1 / (std_dev * math.sqrt(2 * math.pi))) * math.exp(exponent)
def penalty_coeffs(x):
    return 1 - gaussian_function(x)/gaussian_function(0)

In [20]:
search_space_subset_train['distance_penalty'] = search_space_subset_train.Abs_distance.map(penalty_coeffs)
search_space_subset_test['distance_penalty'] = search_space_subset_test.Abs_distance.map(penalty_coeffs)

In [21]:
r2gdist_train = search_space_subset_train.pivot(index='Name', columns='Gene', values='Abs_distance')
r2gdist_test = search_space_subset_test.pivot(index='Name', columns='Gene', values='Abs_distance')

/tmp/ipykernel_1014941/26681771.py:1: PerformanceWarning: The following operation may generate 3299567634 cells in the resulting pandas object.
  r2gdist_train = search_space_subset_train.pivot(index='Name', columns='Gene', values='Abs_distance')


In [22]:
r2gpenalty_train = search_space_subset_train.pivot(index='Name', columns='Gene', values='distance_penalty')
r2gpenalty_test = search_space_subset_test.pivot(index='Name', columns='Gene', values='distance_penalty')

/tmp/ipykernel_1014941/3925202088.py:1: PerformanceWarning: The following operation may generate 3299567634 cells in the resulting pandas object.
  r2gpenalty_train = search_space_subset_train.pivot(index='Name', columns='Gene', values='distance_penalty')


In [23]:
r2gdist_train = r2gdist_train.fillna(0)
r2gdist_test = r2gdist_test.fillna(0)
r2gpenalty_train = r2gpenalty_train.fillna(0)
r2gpenalty_test = r2gpenalty_test.fillna(0)

In [24]:
# Add missing TFs to the datasets
r2gdist_train.loc[:, TFs[~np.isin(TFs, r2gdist_train.columns)]] = 0
r2gdist_test.loc[:, TFs[~np.isin(TFs, r2gdist_test.columns)]] = 0
r2gpenalty_train.loc[:, TFs[~np.isin(TFs, r2gpenalty_train.columns)]] = 0
r2gpenalty_test.loc[:, TFs[~np.isin(TFs, r2gpenalty_test.columns)]] = 0

/tmp/ipykernel_1014941/3167708050.py:2: FutureWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  r2gdist_train.loc[:, TFs[~np.isin(TFs, r2gdist_train.columns)]] = 0
/tmp/ipykernel_1014941/3167708050.py:3: FutureWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  r2gdist_test.loc[:, TFs[~np.isin(TFs, r2gdist_test.columns)]] = 0
/tmp/ipykernel_1014941/3167708050.py:4: FutureWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` o

In [25]:
# Put TFs first
r2gdist_train = r2gdist_train.loc[:, TFs.to_list() + r2gdist_train.columns[~r2gdist_train.columns.isin(TFs)].to_list()]
r2gdist_test = r2gdist_test.loc[:, TFs.to_list() + r2gdist_test.columns[~r2gdist_test.columns.isin(TFs)].to_list()]
r2gpenalty_train = r2gpenalty_train.loc[:, TFs.to_list() + r2gpenalty_train.columns[~r2gpenalty_train.columns.isin(TFs)].to_list()]
r2gpenalty_test = r2gpenalty_test.loc[:, TFs.to_list() + r2gpenalty_test.columns[~r2gpenalty_test.columns.isin(TFs)].to_list()]

In [26]:
# Ensure same order of indices
regions_train = r2gdist_train.index.values.tolist()
genes_train = r2gdist_train.columns.values.tolist()

regions_test = r2gdist_test.index.values.tolist()
genes_test = r2gdist_test.columns.values.tolist()

ad_atac_train = ad_atac[:, regions_train].copy()
ad_rna_train = ad_rna[:, genes_train].copy()
ad_atac_test = ad_atac[:, regions_test].copy()
ad_rna_test = ad_rna[:, genes_test].copy()

In [27]:
from scipy.sparse import coo_array, save_npz
r2gpenalty_coo_train = coo_array(r2gpenalty_train)
r2gpenalty_coo_test = coo_array(r2gpenalty_test)

In [28]:
ad_rna_train.write(DATA_PATH+'raw_exprMat_train.h5ad')
ad_atac_train.write(DATA_PATH+'fragment_matrix_train.h5ad')
ad_rna_test.write(DATA_PATH+'raw_exprMat_test.h5ad')
ad_atac_test.write(DATA_PATH+'fragment_matrix_test.h5ad')
save_npz(DATA_PATH+'r2gpenalty_train.npz', r2gpenalty_coo_train)
save_npz(DATA_PATH+'r2gpenalty_test.npz', r2gpenalty_coo_test)
with open(DATA_PATH+'TFs.txt', 'w') as f:
    for item in TFs:
        f.write("%s\n" % item)   

In [29]:
print("ciao")

ciao


In [30]:
for r in ad_atac_train.var_names:
    _, coords = r.split(':')
    start, end = coords.split('-')
    if (int(end)-int(start))<500:
        print(r)

In [31]:
import scanpy as sc

DATA_PATH = '/lustre1/project/stg_00002/lcb/gpartel/projects/deepSCENIC/data/MM_lines/'
ad_rna_train = sc.read(DATA_PATH+'raw_exprMat_train.h5ad')
ad_rna_test = sc.read(DATA_PATH+'raw_exprMat_test.h5ad')

ad_atac_train = sc.read(DATA_PATH+'fragment_matrix_train.h5ad')
ad_atac_test = sc.read(DATA_PATH+'fragment_matrix_test.h5ad')

In [32]:
ad_rna = sc.concat((ad_rna_train, ad_rna_test[:, ~ad_rna_test.var_names.isin(ad_rna_train.var_names)]), axis=1)
ad_rna

AnnData object with n_obs × n_vars = 936 × 12188
    var: 'n_cells'
    layers: 'log_norm'

In [33]:
ad_atac = sc.concat((ad_atac_train, ad_atac_test), axis=1)
ad_atac

AnnData object with n_obs × n_vars = 936 × 281820

In [34]:
ad_rna.write(DATA_PATH+'raw_exprMat.h5ad')
ad_atac.write(DATA_PATH+'fragment_matrix.h5ad')

In [35]:
from sklearn.model_selection import train_test_split
train_idxs, test_idxs = train_test_split(ad_rna_train.obs.index, test_size=0.2, random_state=42)

In [36]:
ad_atac_train = ad_atac_train[train_idxs]
ad_rna_train = ad_rna_train[train_idxs]
ad_atac_test = ad_atac_test[test_idxs]
ad_rna_test = ad_rna_test[test_idxs]

In [37]:
ad_rna_train.write(DATA_PATH+'raw_exprMat_train.h5ad')
ad_atac_train.write(DATA_PATH+'fragment_matrix_train.h5ad')
ad_rna_test.write(DATA_PATH+'raw_exprMat_test.h5ad')
ad_atac_test.write(DATA_PATH+'fragment_matrix_test.h5ad')

In [38]:
ad_rna = sc.read(DATA_PATH+'raw_exprMat.h5ad')
ad_rna[ad_rna_test.obs.index, ad_rna_train.var_names].write(DATA_PATH+'raw_exprMat_train_eval.h5ad')